
# Hospital EDW — Attribute-Based Access Control (ABAC) Tag Policies

This notebook implements **Approach 1 (Centralized ABAC Tag Policies)** in Databricks Unity Catalog using `CREATE POLICY`.

### Why Approach 1:
1. **Centralized Enforcement**: Policies attach at the `CATALOG` level (`hospital_catalog`).
2. **Automated Protection**: Any column tagged with `info_type` or `financial = true` is automatically masked for restricted groups.
3. **Zero Maintenance**: New tables added to the catalog are governed immediately upon tagging.


## Step 1: Create Governance Schema & Masking Functions

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS hospital_catalog.governance
COMMENT 'Central governance schema for ABAC policy UDFs and access control rules';

In [ ]:
%sql
-- 1.1 Name Masking UDF (Patients & Doctors)
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_name(val STRING)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('clinical_stewards')
    THEN val
  WHEN val IS NULL THEN NULL
  ELSE CONCAT(SUBSTRING(val, 1, 1), '****')
END;

In [ ]:
%sql
-- 1.2 Phone Number Masking UDF
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_phone(phone STRING)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('clinical_stewards')
    THEN phone
  WHEN phone IS NULL THEN NULL
  ELSE CONCAT('***-***-', RIGHT(phone, 4))
END;

In [ ]:
%sql
-- 1.3 Physical Address Masking UDF
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_address(address STRING)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('clinical_stewards')
    THEN address
  WHEN address IS NULL THEN NULL
  ELSE '*** REDACTED ADDRESS ***'
END;

In [ ]:
%sql
-- 1.4 Date of Birth Masking UDF (HIPAA Safe Harbor)
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_dob(dob DATE)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('clinical_stewards')
    THEN dob
  WHEN dob IS NULL THEN NULL
  ELSE MAKE_DATE(YEAR(dob), 1, 1)
END;

In [ ]:
%sql
-- 1.5 Financial Amount Masking UDF
CREATE OR REPLACE FUNCTION hospital_catalog.governance.mask_financial(amount DOUBLE)
RETURN CASE
  WHEN is_account_group_member('hospital_admins') 
    OR is_account_group_member('billing_finance')
    THEN amount
  ELSE NULL
END;

In [ ]:
%sql
-- 1.6 Hospital Branch Isolation Row Filter UDF
CREATE OR REPLACE FUNCTION hospital_catalog.governance.filter_branch(branch STRING)
RETURN 
  is_account_group_member('hospital_admins') OR
  (is_account_group_member('cairo_branch_staff') AND branch = 'CAI') OR
  (is_account_group_member('alexandria_branch_staff') AND branch = 'ALX') OR
  (is_account_group_member('giza_branch_staff') AND branch = 'GIZ');


## Step 2: Apply Column Masks — Name columns (first_name / last_name / staff_name)

Databricks Unity Catalog does **not** support `CREATE POLICY ... FOR COLUMNS MATCH COLUMNS has_tag_value(...)`.  
The correct syntax is `ALTER TABLE … ALTER COLUMN … SET MASK <function>` applied per column.

In [ ]:
%sql
-- 2.1 Name masking — Silver patients
ALTER TABLE hospital_catalog.silver.patients ALTER COLUMN first_name SET MASK hospital_catalog.governance.mask_name;

ALTER TABLE hospital_catalog.silver.patients ALTER COLUMN last_name  SET MASK hospital_catalog.governance.mask_name;

In [ ]:
%sql
-- 2.2 Name masking — Silver doctors
ALTER TABLE hospital_catalog.silver.doctors ALTER COLUMN first_name SET MASK hospital_catalog.governance.mask_name;

ALTER TABLE hospital_catalog.silver.doctors ALTER COLUMN last_name  SET MASK hospital_catalog.governance.mask_name;

In [ ]:
%sql
-- 2.3 Name masking — Gold dim_patient_enterprise
ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN first_name SET MASK hospital_catalog.governance.mask_name;

ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN last_name  SET MASK hospital_catalog.governance.mask_name;

In [ ]:
%sql
-- 2.4 Name masking — Gold dim_doctor_enterprise
ALTER TABLE hospital_catalog.gold.dim_doctor_enterprise ALTER COLUMN first_name SET MASK hospital_catalog.governance.mask_name;

ALTER TABLE hospital_catalog.gold.dim_doctor_enterprise ALTER COLUMN last_name  SET MASK hospital_catalog.governance.mask_name;


## Step 3: Apply Column Masks — Phone Number columns

In [ ]:
%sql
-- 3.1 Phone masking — Silver patients
ALTER TABLE hospital_catalog.silver.patients             ALTER COLUMN contact_no SET MASK hospital_catalog.governance.mask_phone;

-- 3.2 Phone masking — Silver doctors
ALTER TABLE hospital_catalog.silver.doctors              ALTER COLUMN contact_no SET MASK hospital_catalog.governance.mask_phone;

-- 3.3 Phone masking — Gold dim_patient_enterprise
ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN contact_no SET MASK hospital_catalog.governance.mask_phone;

-- 3.4 Phone masking — Gold dim_doctor_enterprise
ALTER TABLE hospital_catalog.gold.dim_doctor_enterprise  ALTER COLUMN contact_no SET MASK hospital_catalog.governance.mask_phone;


## Step 4: Apply Column Masks — Physical Address

In [ ]:
%sql
-- 4.1 Address masking — Silver patients
ALTER TABLE hospital_catalog.silver.patients             ALTER COLUMN address SET MASK hospital_catalog.governance.mask_address;

-- 4.2 Address masking — Gold dim_patient_enterprise
ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN address SET MASK hospital_catalog.governance.mask_address;


## Step 5: Apply Column Masks — Date of Birth (HIPAA Safe Harbor)

In [ ]:
%sql
-- 5.1 DOB masking — Silver patients
ALTER TABLE hospital_catalog.silver.patients             ALTER COLUMN date_of_birth SET MASK hospital_catalog.governance.mask_dob;

-- 5.2 DOB masking — Gold dim_patient_enterprise
ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN date_of_birth SET MASK hospital_catalog.governance.mask_dob;


## Step 6: Apply Column Masks — Financial / Monetary Amount columns

In [ ]:
%sql
-- 6.1 Financial masking — Silver pharmacy_transactions
ALTER TABLE hospital_catalog.silver.pharmacy_transactions ALTER COLUMN total_amount SET MASK hospital_catalog.governance.mask_financial;

-- 6.2 Financial masking — Silver invoices
ALTER TABLE hospital_catalog.silver.invoices ALTER COLUMN gross_amount             SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.silver.invoices ALTER COLUMN discount_amount          SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.silver.invoices ALTER COLUMN insurance_covered_amount SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.silver.invoices ALTER COLUMN net_amount_due           SET MASK hospital_catalog.governance.mask_financial;

-- 6.3 Financial masking — Silver payments
ALTER TABLE hospital_catalog.silver.payments ALTER COLUMN amount_paid SET MASK hospital_catalog.governance.mask_financial;

-- 6.4 Financial masking — Silver insurance_claims
ALTER TABLE hospital_catalog.silver.insurance_claims ALTER COLUMN claim_amount SET MASK hospital_catalog.governance.mask_financial;

-- 6.5 Financial masking — Gold fact_billing
ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN gross_amount             SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN discount_amount          SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN insurance_covered_amount SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN net_amount_due           SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN total_amount_paid        SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN outstanding_balance      SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN total_claim_amount       SET MASK hospital_catalog.governance.mask_financial;
ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN total_insurance_paid     SET MASK hospital_catalog.governance.mask_financial;

-- 6.6 Financial masking — Gold fact_appointments
ALTER TABLE hospital_catalog.gold.fact_appointments ALTER COLUMN payment_amount SET MASK hospital_catalog.governance.mask_financial;


## Step 7: Apply Row Filters — Branch Isolation

In [ ]:
%sql
-- Row filter: Silver tables use _source_branch (ingestion metadata column)
ALTER TABLE hospital_catalog.silver.patients   SET ROW FILTER hospital_catalog.governance.filter_branch ON (_source_branch);
ALTER TABLE hospital_catalog.silver.doctors    SET ROW FILTER hospital_catalog.governance.filter_branch ON (_source_branch);
ALTER TABLE hospital_catalog.silver.encounters SET ROW FILTER hospital_catalog.governance.filter_branch ON (_source_branch);
ALTER TABLE hospital_catalog.silver.admissions SET ROW FILTER hospital_catalog.governance.filter_branch ON (_source_branch);

-- Gold fact tables alias _source_branch -> source_branch
ALTER TABLE hospital_catalog.gold.fact_appointments SET ROW FILTER hospital_catalog.governance.filter_branch ON (source_branch);


## Step 8: Grant Table Privileges to Groups

> **Before running:** Make sure all groups exist in Databricks Account Admin and that you have `METASTORE ADMIN` or `CATALOG OWNER` privilege.


### 8.1 Prerequisite — Verify Groups Exist

In [ ]:
# ── PREREQUISITE CHECK ─────────────────────────────────────────────────────
# Before running the GRANT cells below, ensure all groups exist in your
# Databricks Account Admin (account.azuredatabricks.net or accounts.cloud.databricks.com):
#
#   Group name           | Who belongs
#   -------------------- | --------------------------------------------------
#   data_analysts        | BI developers, Power BI / Tableau service accounts
#   clinical_stewards    | Clinicians and clinical data stewards
#   billing_finance      | Finance and billing team members
#   hospital_admins      | Metastore admins / data platform owners
#   cairo_branch_staff   | Staff whose data scope is the Cairo branch
#   alexandria_branch_staff | Staff whose data scope is the Alexandria branch
#   giza_branch_staff    | Staff whose data scope is the Giza branch
#   researchers          | Research / analytics team
#
# You also need METASTORE ADMIN or CATALOG OWNER privilege to run GRANT ON CATALOG.
#
# The cells below use spark.sql() with try/except so a missing group or
# insufficient privilege on one statement won't block the rest.

REQUIRED_GROUPS = [
    "data_analysts",
    "clinical_stewards",
    "billing_finance",
    "hospital_admins",
    "cairo_branch_staff",
    "alexandria_branch_staff",
    "giza_branch_staff",
]

try:
    existing = [
        r["groupName"]
        for r in spark.sql("SHOW GROUPS").collect()
    ]
    missing = [g for g in REQUIRED_GROUPS if g not in existing]
    if missing:
        print("\u26a0️  MISSING GROUPS — create these in Databricks Account Admin before running GRANT cells:")
        for g in missing:
            print(f"   - {g}")
    else:
        print("\u2705 All required groups exist. Safe to run GRANT cells.")
except Exception as e:
    print(f"Could not list groups (may need metastore admin): {e}")
    print("Proceed only if you are a metastore admin or catalog owner.")


### 8.2 Data Analysts (BI Developers, Power BI, Tableau)


In [ ]:
grants = [
    "GRANT USE CATALOG ON CATALOG hospital_catalog TO `data_analysts`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.gold TO `data_analysts`",
    "GRANT SELECT ON SCHEMA hospital_catalog.gold TO `data_analysts`",
]

for stmt in grants:
    try:
        spark.sql(stmt)
        print(f"\u2705  {stmt}")
    except Exception as e:
        print(f"\u274c  FAILED: {stmt}")
        print(f"        Reason: {e}")


### 8.3 Clinical Stewards & Clinicians


In [ ]:
grants = [
    "GRANT USE CATALOG ON CATALOG hospital_catalog TO `clinical_stewards`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.silver TO `clinical_stewards`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.gold TO `clinical_stewards`",
    "GRANT SELECT ON SCHEMA hospital_catalog.gold TO `clinical_stewards`",
    "GRANT SELECT ON TABLE hospital_catalog.silver.patient_steward_review_queue_enterprise TO `clinical_stewards`",
    "GRANT SELECT ON TABLE hospital_catalog.silver.doctor_steward_review_queue_enterprise TO `clinical_stewards`",
]

for stmt in grants:
    try:
        spark.sql(stmt)
        print(f"\u2705  {stmt}")
    except Exception as e:
        print(f"\u274c  FAILED: {stmt}")
        print(f"        Reason: {e}")


### 8.4 Billing & Finance Team


In [ ]:
grants = [
    "GRANT USE CATALOG ON CATALOG hospital_catalog TO `billing_finance`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.gold TO `billing_finance`",
    "GRANT SELECT ON TABLE hospital_catalog.gold.fact_billing TO `billing_finance`",
    "GRANT SELECT ON TABLE hospital_catalog.gold.dim_payer TO `billing_finance`",
]

for stmt in grants:
    try:
        spark.sql(stmt)
        print(f"\u2705  {stmt}")
    except Exception as e:
        print(f"\u274c  FAILED: {stmt}")
        print(f"        Reason: {e}")


### 8.5 Branch Staff (Cairo, Alexandria, Giza)


In [ ]:
grants = [
    # Cairo Branch Staff
    "GRANT USE CATALOG ON CATALOG hospital_catalog TO `cairo_branch_staff`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.silver TO `cairo_branch_staff`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.gold TO `cairo_branch_staff`",
    "GRANT SELECT ON SCHEMA hospital_catalog.silver TO `cairo_branch_staff`",
    "GRANT SELECT ON SCHEMA hospital_catalog.gold TO `cairo_branch_staff`",
    # Alexandria Branch Staff
    "GRANT USE CATALOG ON CATALOG hospital_catalog TO `alexandria_branch_staff`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.silver TO `alexandria_branch_staff`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.gold TO `alexandria_branch_staff`",
    "GRANT SELECT ON SCHEMA hospital_catalog.silver TO `alexandria_branch_staff`",
    "GRANT SELECT ON SCHEMA hospital_catalog.gold TO `alexandria_branch_staff`",
    # Giza Branch Staff
    "GRANT USE CATALOG ON CATALOG hospital_catalog TO `giza_branch_staff`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.silver TO `giza_branch_staff`",
    "GRANT USE SCHEMA ON SCHEMA hospital_catalog.gold TO `giza_branch_staff`",
    "GRANT SELECT ON SCHEMA hospital_catalog.silver TO `giza_branch_staff`",
    "GRANT SELECT ON SCHEMA hospital_catalog.gold TO `giza_branch_staff`",
]

for stmt in grants:
    try:
        spark.sql(stmt)
        print(f"\u2705  {stmt}")
    except Exception as e:
        print(f"\u274c  FAILED: {stmt}")
        print(f"        Reason: {e}")
